### Better formulations CFLP

A better formulation rather than the standard integer programming formulation for the FLP (with disaggregated constraints) is as follows:

$$
\begin{aligned}
\text{min} \quad & \sum_{i \in I} f_i y_i + \sum_{i \in I} \sum_{j \in J} c_{ij} x_{ij} \\
\text{subject to} \quad & \sum_{i \in I} x_{ij} = 1 \quad \forall j \in J \\
& \sum_{j \in J} d_j x_{ij} \leq q_i \quad \forall i \in I \\
& x_{ij} \leq y_i \quad \forall i \in I,\; \forall j \in J \\
& y_i \in \{0,1\} \quad \forall i \in I \\
& x_{ij} \in \{0,1\} \quad \forall i \in I,\; \forall j \in J
\end{aligned}
$$



In [1]:
%pip install gurobipy
from gurobipy import Model, GRB, quicksum
import numpy as np

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 32.9 MB/s eta 0:00:00


In [2]:
# Number of facilities
facilities = 5

# Number of customers
customers = 5

# Sets
I = range(facilities)
J = range(customers)

# Symmetric shipping cost matrix
c = np.array([[0, 3, 3, 6, 3],
              [3, 0, 4, 5, 5],
              [3, 4, 0, 2, 4],
              [6, 5, 2, 0, 5],
              [3, 5, 4, 5, 0]])

# Facility opening cost
f = np.array([25.0, 25.0, 25.0, 25.0, 25.0])

# Demand
d = np.array([10, 8, 5, 8, 12])

# Capacity
q = np.array([15, 15, 15, 15, 15])

In [6]:
# Define model
m = Model("fixed-charge")

# Decision variables
#x = m.addVars(I, J, vtype=GRB.BINARY, name="x")
x = m.addVars(I, J, name="x")
y = m.addVars(I, vtype=GRB.BINARY, name="y")

# Objective function
m.setObjective(quicksum(f[i]*y[i] for i in I) + quicksum(c[i,j]*x[i,j]*d[j] for i in I for j in J), GRB.MINIMIZE)

# Constraints
m.addConstrs(quicksum(x[i,j] for i in I) == 1 for j in J)
m.addConstrs(quicksum(x[i,j]*d[j] for j in J) <= q[i] for i in I)
m.addConstrs(x[i,j] - y[i] <= 0 for i in I for j in J)

# optimize
m.optimize()

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 35 rows, 30 columns and 100 nonzeros (Min)
Model fingerprint: 0xd41c1457
Model has 25 linear objective coefficients
Variable types: 25 continuous, 5 integer (5 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+01]
  Objective range  [1e+01, 6e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+01]

Presolve time: 0.00s
Presolved: 35 rows, 30 columns, 105 nonzeros
Variable types: 25 continuous, 5 integer (5 binary)
Found heuristic solution: objective 125.0000000

Root relaxation: objective 1.093750e+02, 8 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0   

In [7]:
if m.status == GRB.OPTIMAL:
    print("Optimal solution")
    print("Total cost: ", m.objVal)
    for i in I:
      if y[i].X > 0.1:
          print("Open facility: ", i)
else:
    print("There is not optimal solution")

Optimal solution
Total cost:  110.0
Open facility:  0
Open facility:  1
Open facility:  3
Open facility:  4
